# 01. Tooling

**Tier:** Intermediate. **Why you'd use this:** A small program needs logs you can filter later, and a linter that catches issues a type checker will not. `print` is fine for a scratch cell; it is a weak log for a real run.

**After this lesson you can:** read a log line from a configured logger; choose `logging` over `print` for a program; say what ruff catches that a type checker does not.
About 25 minutes. **Uses:** [Docstrings and hints](../05-functions/04-hints-lesson.ipynb), [pytest](../15-testing/00-pytest-lesson.ipynb)


## Quick review

1. What does a type hint help a reader or checker do?
2. Name one thing pytest gives you over running a file by hand.

<details><summary>Answers</summary>

Expected types for parameters and results. Discovery, isolation, and clear pass/fail reports.
</details>


## The idea

Programs need three kinds of checking and one kind of recording. **Ruff** reads source and flags style issues and likely bugs before you run. A **type checker** reads type hints and reports mismatches without executing the file. **Tests** execute paths you wrote checks for. **Logging** records events at a level (`INFO`, `WARNING`, `ERROR`) so operators can filter noise without editing `print` calls. Hints are ignored at runtime: a wrong argument still runs until something else fails.

<details><summary>Explain more</summary>

Point ruff at a path with `ruff check path`. Point a type checker at the same path when you want hint mismatches. This lesson captures logs in a `StringIO` buffer so demos stay stable. Prefer named loggers (`logging.getLogger(__name__)`) so libraries and apps can configure levels separately.
</details> **cProfile** times function calls while the program runs so you can see which functions dominate wall time before you optimize.


## See it work

Log an event into a string, then show ruff catching a real lint on a temp file while a type mismatch still runs.


In [1]:
import io
import logging
import subprocess
import sys
from pathlib import Path


def logged(message: str) -> str:
    # 1. Send the log to a string, not the screen.
    buffer = io.StringIO()
    log = logging.getLogger("pylearn-worked")
    log.handlers = [logging.StreamHandler(buffer)]
    log.setLevel(logging.INFO)
    log.propagate = False
    # 2. info is the level for a normal event.
    log.info(message)
    return buffer.getvalue()


print("ready" in logged("ready"))

demo = Path("_ruff_demo_unused.py")
demo.write_text("def greet(name):\n    unused = 1\n    return name\n", encoding="utf-8")
result = subprocess.run(
    [sys.executable, "-m", "ruff", "check", str(demo)],
    capture_output=True,
    text=True,
)
print(result.returncode != 0)
print("F841" in result.stdout or "unused" in result.stdout.lower())
demo.unlink()

True
True
True


## Check yourself

Answer these from the example above before you run the cell.


1. Why did `log.info("silent")` produce no text when the level was `WARNING`?
2. What did ruff catch that the type mismatch demo did not need a checker for?
3. Why can tests miss a bug that a type checker would see?

<details><summary>Answers</summary>

1. INFO is below WARNING, so it is filtered. 2. An unused variable (`F841`). 3. If no test calls the mismatched path, tests stay green.
</details>


In [2]:
buffer = io.StringIO()
handler = logging.StreamHandler(buffer)
log = logging.getLogger("pylearn-demo")
log.handlers = [handler]
log.setLevel(logging.INFO)
log.propagate = False
log.info("ready")
print(buffer.getvalue().splitlines()[-1].endswith("ready"))


def add(a: int, b: int) -> int:
    return a + b


print(add("x", "y"))

True
xy


<details><summary>Check your answers</summary>

`True` — the last log line ends with `ready`. Then `xy` — the hints do not stop the call at runtime.
</details>


## Why it works this way

Ruff and type checkers are **static**: they parse or type-check source without running your business logic. That is cheap and catches whole classes of mistakes early, but they only see what the tools model. Tests are **dynamic**: they prove behaviour on the inputs you chose, including integration paths static tools miss. Logging is orthogonal: it is an observability channel with levels and handlers, not a correctness tool. Design reason: separate "is this shaped right?" (hints/checker), "does this behave?" (tests), "what happened in prod?" (logs).

Cost: static tools are fast per file; tests cost runtime and fixture design; logging costs I/O and needs levels so you do not drown in `DEBUG`. Handlers decide where records go (stream, file); filters and levels decide which records survive. Hints cost nothing at runtime in CPython—they live in annotations and are not enforced, which is why `add("x", "y")` returns `xy`. A type checker would reject that call; ruff may stay quiet; a test fails only if you assert on the types or results.

Deep dive: pair with [Docstrings and hints](../05-functions/04-hints-lesson.ipynb) for the runtime story; there is no separate internals lesson for logging. Evidence: a configured logger records `ready`; INFO below WARNING is filtered; `ruff check` fails on an unused variable (`F841`).

Think of the three tools as layers on one change: ruff rejects noisy mistakes in seconds, the type checker rejects impossible calls before merge, and tests reject wrong behaviour on the cases you named. Logging sits beside them so the running system can explain itself after deploy. Skipping any layer is fine for a toy script; skipping all three on a service is how silent failures ship.

`cProfile` instruments calls at runtime: each enter/leave updates counters. That overhead is acceptable for diagnosis and wrong for a final production hot path left profiling forever. Read the cumulative column first; optimize the few functions that own most of the time.


In [3]:
buffer = io.StringIO()
log = logging.getLogger("pylearn-why")
log.handlers = [logging.StreamHandler(buffer)]
log.setLevel(logging.WARNING)
log.propagate = False
log.info("silent")
log.warning("loud")
text = buffer.getvalue()
print("silent" in text)
print("loud" in text)

False
True


Profile a tiny function to see call counts. Real sessions write to a file or `pstats`; here the report goes to a string.


In [4]:
import cProfile
import io
import pstats


def busy(n: int) -> int:
    total = 0
    for i in range(n):
        total += i * i
    return total


buffer = io.StringIO()
profiler = cProfile.Profile()
profiler.enable()
busy(2000)
profiler.disable()
stats = pstats.Stats(profiler, stream=buffer)
stats.sort_stats("cumulative")
stats.print_stats(5)
report = buffer.getvalue()
print("busy" in report)
print("function calls" in report.lower() or "ncalls" in report.lower())

True
True


## When to use it

Use ruff (or another linter) on every change. Use a type checker on libraries and long-lived apps. Use tests for behaviour. Use logging in programs that run unattended; keep `print` for quick scratch or CLIs that write data to stdout. Don't log secrets. Don't replace tests with type hints. Don't set the root logger to `DEBUG` in production without a plan. Use `cProfile` when a program feels slow and you need evidence about which functions dominate before changing code.


## Common mistake

Calling a fake level name on a logger and reading only the last line of the traceback.


In [5]:
import logging

log = logging.getLogger("pylearn-bad")
print(log.nope("x"))

AttributeError: 'Logger' object has no attribute 'nope'

The last line names `AttributeError`. Real levels include `debug`, `info`, `warning`, `error`, `critical`.


In [6]:
import io
import logging

buffer = io.StringIO()
log = logging.getLogger("pylearn-fixed")
log.handlers = [logging.StreamHandler(buffer)]
log.setLevel(logging.INFO)
log.propagate = False
log.error("failed")
print("failed" in buffer.getvalue())

True


## Try it

Change `logged` so it uses `warning` instead of `info`. Does the word `ready` still appear when the logger level stays `INFO`? What level must you set for the warning to show?


## Recap

- Ruff is static lint; type checkers check hints; tests check behaviour.
- Hints are not enforced at runtime.
- `logging` records leveled events you can filter.

- `cProfile` shows which calls dominate time.
**Next:** [practice](01-tooling-practice.ipynb)
